# Christmas Light Estimator — Lights/Cords Segmentation (Colab GPU)

Trains a U-Net to segment **lights** (fascia, class 1) and **cords** (class 2) from house photos.
Labels were extracted locally from the marked-up PDFs (`ml/labeling/extract.py`) and packaged into
`dataset.zip` (images + masks + splits). **This notebook only needs that zip.**

**Before running:** Runtime → Change runtime type → **GPU (T4 is fine)**.

Pipeline: unzip dataset → augment → U-Net(ResNet34) → Dice+Focal → per-class IoU/F1 (+ tolerant
line recall) on held-out **houses** → save best checkpoint to Drive → download for local inference.

> Lights are the priority and the strong signal; cords are sparser — metrics are reported separately.

## 1. Install deps  (torch + CUDA are preinstalled on Colab)

In [ ]:
!pip -q install segmentation-models-pytorch==0.3.4 "albumentations==1.4.18"

In [ ]:
import torch, os, random, numpy as np
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type > GPU"

## 2. Config

In [ ]:
from types import SimpleNamespace
CFG = SimpleNamespace(
    data_dir     = "/content/dataset",
    drive_dir    = "/content/drive/MyDrive/xmas_estimator",  # checkpoints saved here
    img_size     = 768,    # 1024 preserves thin lines better but uses more memory (lower batch)
    batch_size   = 8,      # drop to 4 if you raise img_size to 1024
    epochs       = 40,
    lr           = 1e-3,
    weight_decay = 1e-4,
    encoder      = "resnet34",
    encoder_weights = "imagenet",
    num_classes  = 3,                       # 0=bg 1=lights 2=cords
    class_names  = ["bg", "lights", "cords"],
    num_workers  = 2,
    seed         = 42,
)
random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)
device = "cuda"
CFG

## 3. Get the dataset

Put `dataset.zip` (produced locally by `ml/labeling/extract.py build --zip`) in your Drive at
`MyDrive/xmas_estimator/dataset.zip`, then run this cell. (Or comment the Drive block and use
`google.colab.files.upload()` instead.)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, zipfile
ZIP = f"{CFG.drive_dir}/dataset.zip"
assert os.path.exists(ZIP), f"Place dataset.zip at {ZIP} (or change ZIP)"
os.makedirs(CFG.data_dir, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(CFG.data_dir)
print("contents:", sorted(os.listdir(CFG.data_dir)))

## 4. Dataset & augmentations  (geometry transforms apply to image **and** mask together)

In [ ]:
import json, cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

splits = json.load(open(f"{CFG.data_dir}/splits.json"))
print("train", len(splits["train"]), "| val", len(splits["val"]))

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def make_tf(train):
    geo = [A.LongestMaxSize(CFG.img_size),
           A.PadIfNeeded(CFG.img_size, CFG.img_size,
                         border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0)]
    aug = [A.HorizontalFlip(p=0.5),
           A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=7,
                              border_mode=cv2.BORDER_CONSTANT, p=0.5),
           A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
           A.RandomGamma(p=0.3),
           A.GaussNoise(p=0.15)] if train else []
    return A.Compose(geo + aug + [A.Normalize(MEAN, STD), ToTensorV2()])

class HouseSeg(Dataset):
    def __init__(self, ids, train):
        self.ids, self.tf = ids, make_tf(train)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        sid = self.ids[i]
        img = np.array(Image.open(f"{CFG.data_dir}/images/{sid}.jpg").convert("RGB"))
        msk = np.array(Image.open(f"{CFG.data_dir}/masks/{sid}.png"))
        a = self.tf(image=img, mask=msk)
        return a["image"], a["mask"].long()

train_dl = DataLoader(HouseSeg(splits["train"], True), batch_size=CFG.batch_size, shuffle=True,
                      num_workers=CFG.num_workers, drop_last=True, pin_memory=True)
val_dl   = DataLoader(HouseSeg(splits["val"], False), batch_size=CFG.batch_size, shuffle=False,
                      num_workers=CFG.num_workers, pin_memory=True)
print("batches:", len(train_dl), "/", len(val_dl))

### Sanity check — image + mask alignment (cyan=lights, magenta=cords)

In [ ]:
import matplotlib.pyplot as plt
def denorm(t):
    return (t.permute(1, 2, 0).cpu().numpy() * np.array(STD) + np.array(MEAN)).clip(0, 1)

xb, yb = next(iter(val_dl))
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for i in range(4):
    ax[0, i].imshow(denorm(xb[i])); ax[0, i].axis("off")
    ov = denorm(xb[i]).copy(); m = yb[i].numpy()
    ov[m == 1] = [0, 1, 1]; ov[m == 2] = [1, 0, 1]
    ax[1, i].imshow(ov); ax[1, i].axis("off")
plt.tight_layout(); plt.show()

## 5. Model, loss, optimizer

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(CFG.encoder, encoder_weights=CFG.encoder_weights,
                 in_channels=3, classes=CFG.num_classes).to(device)

dice  = smp.losses.DiceLoss(mode="multiclass")
focal = smp.losses.FocalLoss(mode="multiclass")
def criterion(logits, target):          # Dice+Focal handles the heavy class imbalance
    return dice(logits, target) + focal(logits, target)

opt    = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.epochs)
scaler = torch.cuda.amp.GradScaler()
print("trainable params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 1))

## 6. Metrics — per-class IoU/F1 + tolerant line recall (lines need geometry-aware scoring)

In [ ]:
import torch.nn.functional as F

def conf_update(conf, pred, tgt):
    for c in range(CFG.num_classes):
        p, t = pred == c, tgt == c
        conf[c][0] += (p & t).sum().item()
        conf[c][1] += (p & ~t).sum().item()
        conf[c][2] += (~p & t).sum().item()

def iou_f1(conf):
    out = {}
    for c in range(CFG.num_classes):
        tp, fp, fn = conf[c]
        out[CFG.class_names[c]] = (tp / (tp + fp + fn + 1e-9), 2 * tp / (2 * tp + fp + fn + 1e-9))
    return out

@torch.no_grad()
def tolerant_pr(pred, tgt, c, tol=3):
    k = 2 * tol + 1
    pc, tc = (pred == c).float(), (tgt == c).float()
    pd = F.max_pool2d(pc.unsqueeze(1), k, 1, tol).squeeze(1) > 0   # dilate pred
    td = F.max_pool2d(tc.unsqueeze(1), k, 1, tol).squeeze(1) > 0   # dilate gt
    pcb, tcb = pc.bool(), tc.bool()
    rec = (tcb & pd).sum().item() / max(tcb.sum().item(), 1)       # gt within tol of pred
    pre = (pcb & td).sum().item() / max(pcb.sum().item(), 1)       # pred within tol of gt
    return pre, rec

## 7. Train  (saves the best checkpoint, by **lights F1**, to Drive)

In [ ]:
os.makedirs(CFG.drive_dir, exist_ok=True)
CKPT = f"{CFG.drive_dir}/best_unet_resnet34.pt"
best, hist = -1.0, []

for ep in range(1, CFG.epochs + 1):
    model.train(); tl = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            loss = criterion(model(xb), yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tl += loss.item() * xb.size(0)
    sched.step(); tl /= len(train_dl.dataset)

    model.eval(); vl = 0.0
    conf = [[0, 0, 0] for _ in range(CFG.num_classes)]
    tol = {1: [0.0, 0.0], 2: [0.0, 0.0]}; nb = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            with torch.cuda.amp.autocast():
                logits = model(xb); vl += criterion(logits, yb).item() * xb.size(0)
            pred = logits.argmax(1)
            conf_update(conf, pred, yb)
            for c in (1, 2):
                pre, rec = tolerant_pr(pred, yb, c)
                tol[c][0] += pre; tol[c][1] += rec
            nb += 1
    vl /= len(val_dl.dataset); m = iou_f1(conf)
    print(f"ep{ep:02d}  tl {tl:.3f}  vl {vl:.3f} | "
          f"LIGHTS IoU {m['lights'][0]:.3f} F1 {m['lights'][1]:.3f} tolR {tol[1][1]/nb:.2f} | "
          f"CORDS IoU {m['cords'][0]:.3f} F1 {m['cords'][1]:.3f} tolR {tol[2][1]/nb:.2f}")
    hist.append({"ep": ep, "tl": tl, "vl": vl,
                 "lights_f1": m["lights"][1], "cords_f1": m["cords"][1]})

    if m["lights"][1] > best:
        best = m["lights"][1]
        torch.save({"model_state": model.state_dict(), "cfg": vars(CFG),
                    "mean": MEAN, "std": STD, "epoch": ep, "val_lights_f1": best}, CKPT)
        print("   ✔ saved best ->", CKPT)

print("best lights F1:", round(best, 3))

## 8. Curves

In [ ]:
eps = [h["ep"] for h in hist]
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(eps, [h["tl"] for h in hist], label="train"); plt.plot(eps, [h["vl"] for h in hist], label="val")
plt.title("loss"); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(eps, [h["lights_f1"] for h in hist], label="lights F1")
plt.plot(eps, [h["cords_f1"] for h in hist], label="cords F1")
plt.title("val F1"); plt.legend(); plt.show()

## 9. Qualitative — photo / truth / prediction

In [ ]:
ck = torch.load(CKPT, map_location=device); model.load_state_dict(ck["model_state"]); model.eval()
xb, yb = next(iter(val_dl))
with torch.no_grad():
    pr = model(xb.to(device)).argmax(1).cpu().numpy()
fig, ax = plt.subplots(3, 4, figsize=(16, 10))
for i in range(4):
    img = denorm(xb[i])
    ax[0, i].imshow(img); ax[0, i].set_title("photo"); ax[0, i].axis("off")
    gt = img.copy(); g = yb[i].numpy(); gt[g == 1] = [0, 1, 1]; gt[g == 2] = [1, 0, 1]
    ax[1, i].imshow(gt); ax[1, i].set_title("truth"); ax[1, i].axis("off")
    pd = img.copy(); p = pr[i]; pd[p == 1] = [0, 1, 1]; pd[p == 2] = [1, 0, 1]
    ax[2, i].imshow(pd); ax[2, i].set_title("prediction"); ax[2, i].axis("off")
plt.tight_layout(); plt.show()

## 10. Export checkpoint for local inference

The best checkpoint is already on Drive at `MyDrive/xmas_estimator/best_unet_resnet34.pt`.
Download it (or just sync Drive) and drop it into the local app's `ml/checkpoints/`. The local
backend rebuilds the same `smp.Unet(...)` and loads `model_state` — CPU inference is fine.

In [ ]:
print("checkpoint on Drive:", CKPT)
# from google.colab import files; files.download(CKPT)   # uncomment to download to your machine

## 11. (Optional) Try it on your own photos

Run on fresh houses (e.g. the clean `New Picturs/` set) to see real-world behaviour. These have no
labels, so it's a qualitative check only.

In [ ]:
from google.colab import files
uploaded = files.upload()
tf = make_tf(False)
for name in uploaded:
    img = np.array(Image.open(name).convert("RGB"))
    x = tf(image=img, mask=np.zeros(img.shape[:2], np.uint8))["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        p = model(x).argmax(1)[0].cpu().numpy()
    base = denorm(x[0]); ov = base.copy(); ov[p == 1] = [0, 1, 1]; ov[p == 2] = [1, 0, 1]
    plt.figure(figsize=(9, 6)); plt.imshow(ov); plt.title(name); plt.axis("off"); plt.show()